# Entraînement sur Kaggle (GPU)
Activer le GPU et Internet dans les paramètres du notebook. Les identifiants du remote DVC sont dans les **Kaggle Secrets** (jamais dans ce notebook).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID'] = s.get_secret('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = s.get_secret('AWS_SECRET_ACCESS_KEY')

In [ ]:
BRANCH = 'main'   # branche à exécuter
!git clone https://github.com/<organisation>/vishing-detection.git /kaggle/working/vishing-detection
%cd /kaggle/working/vishing-detection
!git checkout {BRANCH}
!pip install -q -e . dvc dvc-s3 mlflow pyspark

In [ ]:
!dvc pull data/processed data/metadata

In [ ]:
os.environ['MLFLOW_TRACKING_URI'] = 'file:///kaggle/working/mlruns'   # ou serveur distant partagé
!python -m vishing_detection.pipeline.run_stage embeddings

In [ ]:
!python -m vishing_detection.pipeline.run_stage train
!python -m vishing_detection.pipeline.run_stage evaluate

In [ ]:
!dvc push
!cat reports/metrics.json

Ensuite, en local : récupérer `mlruns/` si besoin, puis committer `dvc.lock` et `reports/metrics.json`.